# Control Variable Construction

This notebook merges announcement-date and firm-specific control variables into `financial_event_dataset.csv` to produce `financial_event_dataset_with_cv.csv`.

### Variables added
| Variable | Source | Description |
|---|---|---|
| `suescore` | IBES (`sue_surprise_history`) | Standardized Unexpected Earnings (SUE) — earnings surprise at announcement |
| `analyst_coverage` | IBES (`numest_analyst_coverage`) | Number of analyst estimates (FPI=6, most recent consensus pre-announcement) |
| `roa` | Compustat (`cv_compustat`) | Return on Assets = NIQ / ATQ |
| `book_to_market` | Compustat | Book-to-Market ratio = CEQQ / MKVALTQ |
| `firm_size` | Compustat | Firm size = ln(MKVALTQ) |
| `leverage` | Compustat | Leverage = (DLCQ + DLTTQ) / ATQ |

### Merge strategy — all three sources use `merge_asof` per ticker with a date tolerance
`main['date']` is the **transcript call date** from the LSEG filename. IBES (`anndats` / `ANNDATS_ACT`) and Compustat (`rdq`) record the **earnings announcement / report date**. Two sources of offset exist: (1) firms that announce after market close on Day 0 and hold the call next morning on Day 1 (AIG, MET — systematic +1 day); and (2) cases where IBES records an earlier preliminary release date while the call happens several days later (CVX, UNP, FDX — up to +7 days). A **±10-day tolerance** is used for IBES and **±3 days** for Compustat. Both windows are safe given that consecutive quarterly earnings are ~90 days apart.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
# Walk up from the current working directory until we find the project root
# (the folder that contains both 'data/' and 'program/' subdirectories).
def find_project_root(start: Path = Path.cwd(), max_levels: int = 10) -> Path:
    p = start
    for _ in range(max_levels):
        if (p / 'data').is_dir() and (p / 'program').is_dir():
            return p
        p = p.parent
    raise FileNotFoundError(
        f"Could not find project root (needs 'data/' and 'program/' subdirs) "
        f"starting from {start}"
    )

BASE        = find_project_root()
RAW_DIR     = BASE / 'data' / 'raw'
PROC_DIR    = BASE / 'data' / 'processed'

INPUT_FILE  = PROC_DIR / 'financial_event_dataset.csv'
OUTPUT_FILE = PROC_DIR / 'financial_event_dataset_with_cv.csv'

SUE_FILE    = RAW_DIR / 'sue_surprise_history.csv'
NUMEST_FILE = RAW_DIR / 'numest_analyst_coverage.csv'
CV_FILE     = RAW_DIR / 'cv_compustat.csv'

print('Project root :', BASE)
print('Input        :', INPUT_FILE)
print('Output       :', OUTPUT_FILE)

Project root : /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program
Input        : /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/financial_event_dataset.csv
Output       : /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/financial_event_dataset_with_cv.csv


## 1. Load Data

In [2]:
# ── Main event dataset ───────────────────────────────────────────────────────
main = pd.read_csv(INPUT_FILE)
main['date'] = pd.to_datetime(main['date'])
print(f'Main dataset  : {main.shape[0]:,} rows × {main.shape[1]} cols')

# ── SUE ──────────────────────────────────────────────────────────────────────
sue_raw = pd.read_csv(SUE_FILE)
sue_raw['anndats'] = pd.to_datetime(sue_raw['anndats'])
print(f'SUE raw       : {sue_raw.shape[0]:,} rows')

# ── Analyst coverage ─────────────────────────────────────────────────────────
numest_raw = pd.read_csv(NUMEST_FILE)
numest_raw['STATPERS']   = pd.to_datetime(numest_raw['STATPERS'])
numest_raw['ANNDATS_ACT'] = pd.to_datetime(numest_raw['ANNDATS_ACT'])
print(f'NUMEST raw    : {numest_raw.shape[0]:,} rows')

# ── Compustat fundamentals ───────────────────────────────────────────────────
cv_raw = pd.read_csv(CV_FILE)
cv_raw['rdq']      = pd.to_datetime(cv_raw['rdq'])
cv_raw['datadate'] = pd.to_datetime(cv_raw['datadate'])
print(f'Compustat raw : {cv_raw.shape[0]:,} rows')

Main dataset  : 1,547 rows × 48 cols
SUE raw       : 1,818 rows
NUMEST raw    : 23,135 rows
Compustat raw : 1,666 rows


### 1b. Remove known bad / duplicate entries

In [3]:
# ── Known bad / duplicate transcript entries ──────────────────────────────────
# Each entry is a (ticker, date) pair that should be excluded, with the reason.
#
# UNP  2025-07-29  Q2 2025
#   The legitimate UNP Q2 2025 earnings call is 2025-07-24
#   (transcript ID 139645850134). The 2025-07-29 row (ID 137095478969) is a
#   spurious duplicate — its transcript ID is anomalously low relative to all
#   other 2025 transcripts and the date does not correspond to an official UNP
#   earnings call.
# ─────────────────────────────────────────────────────────────────────────────
BAD_ENTRIES = [
    ('UNP', '2025-07-29'),   # spurious duplicate of Q2 2025 earnings call
]

before = len(main)
for ticker, date in BAD_ENTRIES:
    mask = (main['ticker'] == ticker) & (main['date'] == pd.Timestamp(date))
    main = main[~mask].copy()

dropped = before - len(main)
print(f'Dropped {dropped} known bad entr{"y" if dropped == 1 else "ies"}:')
for ticker, date in BAD_ENTRIES:
    print(f'  {ticker}  {date}')
print(f'Remaining rows: {len(main):,}')

Dropped 1 known bad entry:
  UNP  2025-07-29
Remaining rows: 1,546


## 2. SUE Score

Filter to quarterly EPS surprise records (`FISCALP == 'QTR'`) and merge on the current ticker (`oftic`) and the actual announcement date (`anndats`).

In [4]:
# Keep quarterly EPS surprises only; use oftic (current ticker symbol)
sue = (
    sue_raw
    .query("FISCALP == 'QTR'")
    [['oftic', 'anndats', 'suescore']]
    .rename(columns={'oftic': 'tic'})
    .drop_duplicates(subset=['tic', 'anndats'])
    .sort_values('anndats')
)

print(f'SUE (QTR only): {len(sue):,} records across {sue["tic"].nunique()} tickers')

# ── Why merge_asof with ±10-day tolerance? ────────────────────────────────────
# main['date'] = transcript call date (from LSEG filename).
# sue['anndats'] = earnings press-release date recorded by IBES.
# Two sources of offset:
#   • Systematic +1 day (AIG, MET, BMY, NOW): company announces after market
#     close on Day 0; call transcript is dated Day 1.
#   • Larger gaps (CVX +5, UNP +5, FDX +7): likely IBES recording the
#     preliminary/after-hours release date vs. the next trading-day call.
# ±10 days is safe because consecutive quarterly earnings are ~90 days apart,
# so there is no risk of matching to a different quarter's announcement.
# ─────────────────────────────────────────────────────────────────────────────
main_sorted = main.sort_values('date')

sue_chunks = []
for ticker, grp in main_sorted.groupby('ticker'):
    sue_ticker = sue[sue['tic'] == ticker]
    if sue_ticker.empty:
        sue_chunks.append(grp)
        continue
    chunk = pd.merge_asof(
        grp,
        sue_ticker[['anndats', 'suescore']],
        left_on   = 'date',
        right_on  = 'anndats',
        direction = 'nearest',
        tolerance = pd.Timedelta('10 days')
    ).drop(columns='anndats')
    sue_chunks.append(chunk)

main = pd.concat(sue_chunks).sort_values(['ticker', 'date']).reset_index(drop=True)

hit = main['suescore'].notna().sum()
print(f'SUE merge  →  {hit:,} / {len(main):,} matched  ({hit/len(main)*100:.1f}%)')

SUE (QTR only): 1,818 records across 100 tickers
SUE merge  →  1,535 / 1,546 matched  (99.3%)


## 3. Analyst Coverage

Use the **current-quarter** consensus (`FPI = 6`). For each firm-announcement pair, take the most recent statistical period (`STATPERS`) that falls **before** the actual announcement date — this gives the last available analyst count prior to the earnings release.

In [5]:
# Filter to current-quarter estimates (FPI=6) and pre-announcement periods.
# The STATPERS filter uses IBES's own ANNDATS_ACT (correct — we want estimates
# published before the IBES announcement date, regardless of the offset).
numest = (
    numest_raw
    .query("FPI == 6")
    .loc[lambda df: df['STATPERS'] < df['ANNDATS_ACT']]
    .copy()
)

# Keep the most recent STATPERS for each (firm, announcement date)
numest_closest = (
    numest
    .sort_values('STATPERS')
    .groupby(['oftic', 'ANNDATS_ACT'], as_index=False)
    .last()
    [['oftic', 'ANNDATS_ACT', 'NUMEST']]
    .rename(columns={'oftic': 'tic', 'ANNDATS_ACT': 'anndats_act'})
    .sort_values('anndats_act')
)

print(f'NUMEST (FPI=6, pre-ann): {len(numest_closest):,} records')

# ── Same ±10-day tolerance as SUE (see SUE cell for rationale) ────────────────
numest_chunks = []
for ticker, grp in main.sort_values('date').groupby('ticker'):
    nc = numest_closest[numest_closest['tic'] == ticker]
    if nc.empty:
        numest_chunks.append(grp)
        continue
    chunk = pd.merge_asof(
        grp,
        nc[['anndats_act', 'NUMEST']],
        left_on   = 'date',
        right_on  = 'anndats_act',
        direction = 'nearest',
        tolerance = pd.Timedelta('10 days')
    ).drop(columns='anndats_act').rename(columns={'NUMEST': 'analyst_coverage'})
    numest_chunks.append(chunk)

main = pd.concat(numest_chunks).sort_values(['ticker', 'date']).reset_index(drop=True)

hit = main['analyst_coverage'].notna().sum()
print(f'NUMEST merge →  {hit:,} / {len(main):,} matched  ({hit/len(main)*100:.1f}%)')

NUMEST (FPI=6, pre-ann): 1,894 records
NUMEST merge →  1,544 / 1,546 matched  (99.9%)


## 4. Compustat Control Variables

### 4a. Compute ratios

| Variable | Formula |
|---|---|
| `roa` | NIQ / ATQ |
| `book_to_market` | CEQQ / MKVALTQ |
| `firm_size` | ln(MKVALTQ) |
| `leverage` | (DLCQ + DLTTQ) / ATQ |

In [6]:
cv = cv_raw.sort_values(['tic', 'datadate']).copy()

# Financial ratios
cv['roa']            = cv['niq']  / cv['atq']
cv['book_to_market'] = cv['ceqq'] / cv['mkvaltq']
cv['firm_size']      = np.log(cv['mkvaltq'])

# Leverage = (short-term debt + long-term debt) / total assets.
# Two NaN cases in Compustat, both treated as zero debt:
#   1. dlcq is NaN but dlttq is present — firm does not separately report the
#      current portion of long-term debt (common for financial firms like AIG).
#   2. Both dlcq and dlttq are NaN — firm carries no debt at all and does not
#      report debt fields (e.g. ISRG / Intuitive Surgical, which is debt-free).
# In both cases filling NaN → 0 is economically correct, not an imputation.
cv['leverage'] = (cv['dlcq'].fillna(0) + cv['dlttq'].fillna(0)) / cv['atq']

# Keep only columns needed for merge
CV_COLS = ['tic', 'rdq', 'roa', 'book_to_market', 'firm_size', 'leverage']
cv_merge = cv[CV_COLS].sort_values('rdq').copy()

print('Sample Compustat ratios (AAPL):')
display(
    cv[cv['tic']=='AAPL']
    [['tic', 'datadate', 'rdq', 'roa', 'book_to_market', 'firm_size', 'leverage']]
    .tail(8)
    .round(4)
    .reset_index(drop=True)
)
print()
print('ISRG leverage (debt-free firm, should be 0.0):')
display(
    cv[cv['tic']=='ISRG']
    [['tic', 'datadate', 'dlcq', 'dlttq', 'atq', 'leverage']]
    .tail(4)
    .round(4)
    .reset_index(drop=True)
)

Sample Compustat ratios (AAPL):


,tic,datadate,rdq,roa,book_to_market,firm_size,leverage
0,AAPL,2024-03-31,2024-05-02,0.0701,0.0282,14.7825,0.3100
1,AAPL,2024-06-30,2024-08-01,0.0647,0.0208,14.9806,0.3055
2,AAPL,2024-09-30,2024-10-31,0.0404,0.0162,15.0746,0.3262
3,AAPL,2024-12-31,2025-01-30,0.1056,0.0177,15.1417,0.2813
4,AAPL,2025-03-31,2025-05-01,0.0748,0.0201,15.0150,0.2964
5,AAPL,2025-06-30,2025-07-31,0.0707,0.0216,14.9300,0.3068
6,AAPL,2025-09-30,2025-10-30,0.0765,0.0196,15.1404,0.3128
7,AAPL,2025-12-31,2026-01-29,0.1110,0.0221,15.2011,0.2386



ISRG leverage (debt-free firm, should be 0.0):


,tic,datadate,dlcq,dlttq,atq,leverage
0,ISRG,2025-03-31,NaN,NaN,19220.4,0.0000
1,ISRG,2025-06-30,NaN,NaN,20163.2,0.0000
2,ISRG,2025-09-30,NaN,NaN,19351.8,0.0000
3,ISRG,2025-12-31,39.0,131.9,20458.7,0.0084


### 4b. Merge with main dataset

Use `pd.merge_asof` (nearest match on `rdq`) per ticker with a **±3-day tolerance** to handle minor discrepancies between the transcript call date stored in the main dataset and the official report date (`rdq`) in Compustat.

In [7]:
main_sorted = main.sort_values('date').copy()

merged_chunks = []
for ticker, grp in main_sorted.groupby('ticker'):
    cv_ticker = cv_merge[cv_merge['tic'] == ticker]
    if cv_ticker.empty:
        merged_chunks.append(grp)          # no CV data → keep rows, NaN for ratios
        continue
    chunk = pd.merge_asof(
        grp,
        cv_ticker.drop(columns='tic'),
        left_on   = 'date',
        right_on  = 'rdq',
        direction = 'nearest',
        tolerance = pd.Timedelta('3 days')
    ).drop(columns='rdq')
    merged_chunks.append(chunk)

main = pd.concat(merged_chunks).sort_values(['ticker', 'date']).reset_index(drop=True)

hit = main['roa'].notna().sum()
print(f'Compustat merge → {hit:,} / {len(main):,} matched  ({hit/len(main)*100:.1f}%)')

Compustat merge → 1,545 / 1,546 matched  (99.9%)


## 5. Merge Diagnostics

In [8]:
cv_cols  = ['roa', 'book_to_market', 'firm_size', 'leverage']
ann_cols = ['suescore', 'analyst_coverage']
all_new  = ann_cols + cv_cols

print('=== Coverage summary ===')
print(f'{"Variable":<20}  {"Non-null":>8}  {"Missing":>8}  {"Coverage":>9}')
print('-' * 52)
for col in all_new:
    n     = main[col].notna().sum()
    miss  = len(main) - n
    pct   = n / len(main) * 100
    print(f'{col:<20}  {n:>8,}  {miss:>8,}  {pct:>8.1f}%')

print()
print('=== Descriptive statistics ===')
display(main[all_new].describe().round(4))

print()
print('=== Tickers with no SUE / NUMEST data (IBES coverage gap) ===')
miss_sue = main.groupby('ticker')['suescore'].apply(lambda x: x.isna().all())
print(sorted(miss_sue[miss_sue].index.tolist()))

=== Coverage summary ===
Variable              Non-null   Missing   Coverage
----------------------------------------------------
suescore                 1,535        11      99.3%
analyst_coverage         1,544         2      99.9%
roa                      1,545         1      99.9%
book_to_market           1,546         0     100.0%
firm_size                1,546         0     100.0%
leverage                 1,546         0     100.0%

=== Descriptive statistics ===


,suescore,analyst_coverage,roa,book_to_market,firm_size,leverage
count,1535.0000,1544.0000,1545.0000,1546.0000,1546.0000,1546.0000
mean,2.1821,21.0894,0.0222,0.2887,12.0850,0.3327
std,4.0100,7.1546,0.0264,0.3344,0.8995,0.1996
min,-33.9626,3.0000,-0.1807,-0.2507,9.5087,0.0000
25%,0.5520,16.0000,0.0074,0.0766,11.5359,0.1893
50%,1.7091,20.0000,0.0177,0.1736,11.9788,0.3082
75%,3.3235,26.0000,0.0319,0.4234,12.4795,0.4366
max,68.5166,47.0000,0.2117,2.4138,15.4091,0.9750



=== Tickers with no SUE / NUMEST data (IBES coverage gap) ===
[]


## 6. Save Output

In [9]:
main.to_csv(OUTPUT_FILE, index=False)

print(f'Saved → {OUTPUT_FILE}')
print(f'Shape : {main.shape[0]:,} rows × {main.shape[1]} cols')
print(f'New columns added: {all_new}')
print()
print('Final column list:')
for i, c in enumerate(main.columns):
    print(f'  [{i:02d}] {c}')

Saved → /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/financial_event_dataset_with_cv.csv
Shape : 1,546 rows × 54 cols
New columns added: ['suescore', 'analyst_coverage', 'roa', 'book_to_market', 'firm_size', 'leverage']

Final column list:
  [00] company
  [01] ticker
  [02] date
  [03] file
  [04] PERMNO
  [05] Ticker
  [06] IssuerNm
  [07] event_trading_day
  [08] next_call_date
  [09] car_m1_p1
  [10] n_days_car
  [11] long_run_start
  [12] long_run_end
  [13] long_run_abret
  [14] n_days_long_run
  [15] total_words
  [16] pres_words
  [17] qa_words
  [18] core_hits_total
  [19] core_hits_pres
  [20] core_hits_qa
  [21] adj_hits_total
  [22] adj_hits_pres
  [23] adj_hits_qa
  [24] lm_positive_total
  [25] lm_negative_total
  [26] lm_uncertainty_count_total
  [27] lm_tone_total
  [28] lm_uncertainty_total
  [29] lm_positive_pres
  [30] lm_negative_pres
  [31] lm_uncertainty_count_pres
  [32] lm_tone_pres
  [33] lm_uncertainty_pres
  [34] lm_positive_qa
  [35] 

## 7. Extract Missing-Value Entries

Two causes account for all remaining gaps:

| Cause | Columns affected | Root reason |
|---|---|---|
| **B — IBES coverage gap** | `suescore`, `analyst_coverage` | Firm not covered in the IBES extract |
| **C — Compustat gap** | `roa`, `book_to_market`, `firm_size`, `leverage` | Report date falls outside the Compustat window (future or missing) |

In [10]:
# ── Column groups ─────────────────────────────────────────────────────────────
CV_COLS        = ['suescore', 'analyst_coverage', 'roa',
                  'book_to_market', 'firm_size', 'leverage']
IBES_COLS      = ['suescore', 'analyst_coverage']          # Cause B
COMPUSTAT_COLS = ['roa', 'book_to_market', 'firm_size', 'leverage']  # Cause C
ID_COLS        = ['ticker', 'date', 'fiscal_period']

# ── Reload the saved file (so this section is self-contained) ─────────────────
data = pd.read_csv(OUTPUT_FILE, parse_dates=['date'])

# ─────────────────────────────────────────────────────────────────────────────
# Helper: tag each row with one or more missing-value causes
# ─────────────────────────────────────────────────────────────────────────────
def classify_missing(row):
    causes = []
    # B: IBES not available for this firm
    if any(pd.isna(row[c]) for c in IBES_COLS):
        causes.append('B_ibes_gap')
    # C: Compustat data outside available window
    if any(pd.isna(row[c]) for c in COMPUSTAT_COLS):
        causes.append('C_compustat_gap')
    return ' | '.join(causes) if causes else None

data['_missing_cause'] = data.apply(classify_missing, axis=1)

# ─────────────────────────────────────────────────────────────────────────────
# 7a. High-level summary table
# ─────────────────────────────────────────────────────────────────────────────
print('=== 7a. Missing-value count per column ===')
print(f'{"Column":<22}  {"Missing":>8}  {"of Total":>9}  {"Pct":>7}')
print('-' * 52)
for col in CV_COLS:
    n_miss = data[col].isna().sum()
    pct    = n_miss / len(data) * 100
    print(f'{col:<22}  {n_miss:>8,}  {len(data):>9,}  {pct:>6.1f}%')

print()
print('=== 7b. Rows affected by each cause ===')
cause_summary = (
    data[data['_missing_cause'].notna()]['_missing_cause']
    .str.split(' | ', regex=False)
    .explode()
    .value_counts()
    .rename('n_rows')
)
print(cause_summary.to_string())

=== 7a. Missing-value count per column ===
Column                   Missing   of Total      Pct
----------------------------------------------------
suescore                      11      1,546     0.7%
analyst_coverage               2      1,546     0.1%
roa                            1      1,546     0.1%
book_to_market                 0      1,546     0.0%
firm_size                      0      1,546     0.0%
leverage                       0      1,546     0.0%

=== 7b. Rows affected by each cause ===
_missing_cause
B_ibes_gap         11
C_compustat_gap     1


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 7c. Extract each missing-value group
# ─────────────────────────────────────────────────────────────────────────────

# Cause B — IBES gap (suescore and/or analyst_coverage missing)
missing_B = data[
    data[IBES_COLS].isna().any(axis=1)
][ID_COLS + IBES_COLS + ['_missing_cause']].copy()

# Cause C — Compustat gap (roa, b/m, firm_size, or leverage missing)
missing_C = data[
    data[COMPUSTAT_COLS].isna().any(axis=1)
][ID_COLS + COMPUSTAT_COLS + ['_missing_cause']].copy()

# All rows with any missing CV
missing_all = data[
    data[CV_COLS].isna().any(axis=1)
][ID_COLS + CV_COLS + ['_missing_cause']].copy()

# ── Print quick views ──────────────────────────────────────────────────────────
print(f'=== Cause B — IBES coverage gap  ({len(missing_B):,} rows) ===')
b_tickers = missing_B.groupby('ticker')[IBES_COLS].apply(lambda g: g.isna().all()).all(axis=1)
print('  Fully absent from IBES:', sorted(b_tickers[b_tickers].index.tolist()))
print()
print(missing_B.groupby('ticker').size().rename('n_missing').sort_values(ascending=False).to_string())

print()
print(f'=== Cause C — Compustat gap  ({len(missing_C):,} rows) ===')
print(missing_C[ID_COLS + COMPUSTAT_COLS].to_string(index=False))

print()
print(f'=== ALL missing entries combined  ({len(missing_all):,} rows) ===')

=== Cause B — IBES coverage gap  (11 rows) ===
  Fully absent from IBES: ['META']

ticker
META    2
ABT     1
BAC     1
BK      1
C       1
JNJ     1
JPM     1
UNH     1
USB     1
WFC     1

=== Cause C — Compustat gap  (1 rows) ===
ticker       date fiscal_period  roa  book_to_market  firm_size  leverage
   HON 2025-10-23       Q3 2025  NaN        0.125573  11.802933  0.472274

=== ALL missing entries combined  (12 rows) ===


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 7d. Export missing-value report to CSV
# ─────────────────────────────────────────────────────────────────────────────
MISSING_REPORT = PROC_DIR / 'missing_cv_entries.csv'

missing_all.to_csv(MISSING_REPORT, index=False)

print(f'Saved missing entries → {MISSING_REPORT}')
print(f'Shape: {missing_all.shape[0]:,} rows × {missing_all.shape[1]} cols')
print()
print('Cause breakdown:')
print(
    missing_all['_missing_cause']
    .str.split(' | ', regex=False)   # regex=False: treat '|' as a literal char
    .explode()
    .value_counts()
    .rename('rows')
    .to_string()
)

Saved missing entries → /Users/dbduy/Desktop/provjp/BAM/Thesis/Data & Program/data/processed/missing_cv_entries.csv
Shape: 12 rows × 10 cols

Cause breakdown:
_missing_cause
B_ibes_gap         11
C_compustat_gap     1
